# A* Algorithm: Treasure Map Simulator

A* finds a cheap path from one start point to one goal. It is like Dijkstra's algorithm with a sense of direction.

The algorithm grew out of 1960s work on robot planning and graph search, where exploring every possible route was too expensive. Today A*-style search appears in games, robotics, maps, puzzle solvers, and navigation systems that need a good path quickly.

In this notebook, you will build it with small objects: spots, walls, a grid world, and a runner that searches for treasure.

<details>
<summary>Big idea</summary>

A* scores each frontier spot with `f = g + h`: real cost so far plus a heuristic guess to the goal.

</details>

## 1. The Mental Model

A grid can act like a graph:

- **Spot**: one coordinate, like `(2, 1)`
- **Wall**: a blocked spot
- **Neighbor**: a nearby walkable spot
- **g cost**: cost from the start
- **h cost**: guessed cost to the goal
- **f cost**: `g + h`, the score A* uses

A* repeats one move: pick the frontier spot with the lowest `f` score, then improve its neighbors.

<details>
<summary>Hint: what makes a heuristic good?</summary>

A good heuristic points toward the goal without overestimating the real remaining cost. On a simple grid, Manhattan distance is a solid choice.

</details>

## 2. Build the Objects

Implementation plan:

1. `Spot` stores a row and column.
2. `GridWorld` reads an ASCII map with `S`, `G`, `.`, and `#`.
3. `SearchStep` records snapshots for replay.
4. `AStarRunner` owns the pathfinding logic.

<details>
<summary>Implementation hint</summary>

Use a priority queue for the frontier. The priority is `g_cost + heuristic`. If the heuristic returns `0`, A* behaves like Dijkstra.

</details>

In [ ]:
from dataclasses import dataclass, field
from heapq import heappop, heappush
from math import inf
from typing import Callable


### Define a Spot

- 

In [ ]:

@dataclass(frozen=True)
class Spot:
    row: int
    col: int

    def manhattan(self, other: "Spot") -> int:
        return abs(self.row - other.row) + abs(self.col - other.col)

    def __str__(self) -> str:
        return f"({self.row}, {self.col})"


### Create the Grid World 
- 

In [ ]:

@dataclass
class GridWorld:
    rows: list[str]
    start: Spot = field(init=False)
    goal: Spot = field(init=False)
    walls: set[Spot] = field(init=False, default_factory=set)

    def __post_init__(self) -> None:
        if not self.rows:
            raise ValueError("GridWorld needs at least one row.")

        width = len(self.rows[0])
        if any(len(row) != width for row in self.rows):
            raise ValueError("Every grid row must have the same width.")

        found_start = None
        found_goal = None
        self.walls = set()

        for row_index, row in enumerate(self.rows):
            for col_index, marker in enumerate(row):
                spot = Spot(row_index, col_index)
                if marker == "S":
                    found_start = spot
                elif marker == "G":
                    found_goal = spot
                elif marker == "#":
                    self.walls.add(spot)

        if found_start is None or found_goal is None:
            raise ValueError("GridWorld needs one S start and one G goal.")

        self.start = found_start
        self.goal = found_goal

    @property
    def height(self) -> int:
        return len(self.rows)

    @property
    def width(self) -> int:
        return len(self.rows[0])

    def neighbors(self, spot: Spot) -> list[Spot]:
        candidates = [
            Spot(spot.row - 1, spot.col),
            Spot(spot.row + 1, spot.col),
            Spot(spot.row, spot.col - 1),
            Spot(spot.row, spot.col + 1),
        ]
        return [candidate for candidate in candidates if self.is_walkable(candidate)]

    def is_walkable(self, spot: Spot) -> bool:
        in_bounds = 0 <= spot.row < self.height and 0 <= spot.col < self.width
        return in_bounds and spot not in self.walls

    def draw(
        self,
        path: list[Spot] | None = None,
        visited: set[Spot] | None = None,
        frontier: set[Spot] | None = None,
    ) -> str:
        path_set = set(path or [])
        visited = visited or set()
        frontier = frontier or set()
        lines = []

        for row_index in range(self.height):
            markers = []
            for col_index in range(self.width):
                spot = Spot(row_index, col_index)
                if spot == self.start:
                    markers.append("S")
                elif spot == self.goal:
                    markers.append("G")
                elif spot in self.walls:
                    markers.append("#")
                elif spot in path_set:
                    markers.append("*")
                elif spot in visited:
                    markers.append("x")
                elif spot in frontier:
                    markers.append("?")
                else:
                    markers.append(".")
            lines.append("".join(markers))

        return "\n".join(lines)


### Define the search step

-

In [ ]:

@dataclass
class SearchStep:
    current: Spot
    action: str
    g_costs: dict[Spot, float]
    frontier: list[tuple[float, Spot]]
    visited: set[Spot]

### Define the A* runner

- 

In [ ]:


class AStarRunner:
    def __init__(self, world: GridWorld, heuristic: Callable[[Spot], float] | None = None):
        self.world = world
        self.heuristic = heuristic or (lambda spot: spot.manhattan(world.goal))

    def find_path(self) -> tuple[list[Spot], dict[Spot, float], list[SearchStep]]:
        start = self.world.start
        goal = self.world.goal
        frontier: list[tuple[float, int, Spot]] = []
        came_from: dict[Spot, Spot | None] = {start: None}
        g_costs: dict[Spot, float] = {start: 0}
        visited: set[Spot] = set()
        steps: list[SearchStep] = []
        counter = 0

        heappush(frontier, (self.heuristic(start), counter, start))

        while frontier:
            _, _, current = heappop(frontier)
            if current in visited:
                continue

            visited.add(current)
            steps.append(self._snapshot(current, "settle", g_costs, frontier, visited))

            if current == goal:
                return self._rebuild_path(came_from, goal), g_costs, steps

            for neighbor in self.world.neighbors(current):
                candidate = g_costs[current] + 1
                if candidate < g_costs.get(neighbor, inf):
                    came_from[neighbor] = current
                    g_costs[neighbor] = candidate
                    counter += 1
                    f_score = candidate + self.heuristic(neighbor)
                    heappush(frontier, (f_score, counter, neighbor))
                    steps.append(self._snapshot(current, f"improve {neighbor} to f={f_score}", g_costs, frontier, visited))

        return [], g_costs, steps

    def _rebuild_path(self, came_from: dict[Spot, Spot | None], goal: Spot) -> list[Spot]:
        path = []
        current: Spot | None = goal

        while current is not None:
            path.append(current)
            current = came_from[current]

        return path[::-1]

    def _snapshot(
        self,
        current: Spot,
        action: str,
        g_costs: dict[Spot, float],
        frontier: list[tuple[float, int, Spot]],
        visited: set[Spot],
    ) -> SearchStep:
        visible_frontier = [(f_score, spot) for f_score, _, spot in frontier]
        visible_frontier = sorted(visible_frontier, key=lambda item: (item[0], item[1].row, item[1].col))
        return SearchStep(
            current=current,
            action=action,
            g_costs=g_costs.copy(),
            frontier=visible_frontier,
            visited=visited.copy(),
        )

## 3. Create a Tiny Maze

The map uses simple symbols: `S` is the start, `G` is the goal, `#` is a wall, and `.` is open ground.

<details>
<summary>Hint: where is the graph?</summary>

Each open spot is a node. Each move up, down, left, or right is an edge with cost `1`.

</details>

In [2]:
maze_text = [
    "....#....",
    ".##.#.##.",
    ".S..#...G",
    ".##...##.",
    "....#....",
]

world = GridWorld(maze_text)
print(world.draw())

....#....
.##.#.##.
.S..#...G
.##...##.
....#....


## 4. Run A*

Start at `S` and search for `G`. The runner returns three things:

- `path`: the final route
- `g_costs`: real cost from start to each reached spot
- `steps`: snapshots for replaying the search

<details>
<summary>Quick check</summary>

The path should bend around the wall instead of trying to walk straight through it.

</details>

In [3]:
runner = AStarRunner(world)
path, g_costs, steps = runner.find_path()

print("Path cost:", int(g_costs[world.goal]))
print("Path:", " -> ".join(str(spot) for spot in path))
print()
print(world.draw(path=path))

Path cost: 9
Path: (2, 1) -> (2, 2) -> (2, 3) -> (3, 3) -> (3, 4) -> (3, 5) -> (2, 5) -> (2, 6) -> (2, 7) -> (2, 8)

....#....
.##.#.##.
.S**#***G
.##***##.
....#....


## 5. Replay the Search

The replay prints a few snapshots so you can see A* balancing real cost and goal direction.

<details>
<summary>Hint: what are `x` and `?`?</summary>

`x` means already visited. `?` means still waiting in the frontier.

</details>

**Trace model.** Define `SearchReplay`, the structure used to capture replayable algorithm state.


In [ ]:
class SearchReplay:
    def __init__(self, world: GridWorld, steps: list[SearchStep]):
        self.world = world
        self.steps = steps

    def show(self, limit: int | None = None) -> None:
        selected_steps = self.steps if limit is None else self.steps[:limit]

        for number, step in enumerate(selected_steps, start=1):
            g_score = step.g_costs.get(step.current, inf)
            h_score = step.current.manhattan(self.world.goal)
            frontier_spots = {spot for _, spot in step.frontier}
            print(f"Step {number}: {step.current} | {step.action}")
            print(f"  g={g_score}, h={h_score}, f={g_score + h_score}")
            print("  frontier:", self._format_frontier(step.frontier))
            print(self.world.draw(visited=step.visited, frontier=frontier_spots))
            print()

    def _format_frontier(self, frontier: list[tuple[float, Spot]]) -> str:
        shown = frontier[:5]
        text = ", ".join(f"{spot}:{int(score)}" for score, spot in shown)
        if len(frontier) > len(shown):
            text += f", +{len(frontier) - len(shown)} more"
        return text or "empty"


**Example state.** Create `replay`, the concrete values used in the next run.


In [ ]:
replay = SearchReplay(world, steps)

replay.show(limit=6)


## 6. Your Experiments

Try changing one thing at a time:

- Move the `#` walls
- Move `S` or `G`
- Compare A* with a zero heuristic
- Make a bigger maze

<details>
<summary>Challenge</summary>

Predict whether A* will visit fewer spots than the zero-heuristic version. Then compare the step counts.

</details>

In [5]:
experiment_map = [
    "....#....",
    ".##...##.",
    ".S..#...G",
    ".##...##.",
    ".........",
]

experiment_world = GridWorld(experiment_map)
experiment_runner = AStarRunner(experiment_world)
experiment_path, experiment_costs, experiment_steps = experiment_runner.find_path()

zero_heuristic_runner = AStarRunner(experiment_world, heuristic=lambda spot: 0)
_, zero_costs, zero_steps = zero_heuristic_runner.find_path()

print("A* cost:", int(experiment_costs[experiment_world.goal]))
print("A* steps:", len(experiment_steps))
print("Zero-heuristic steps:", len(zero_steps))
print("Same shortest cost:", experiment_costs[experiment_world.goal] == zero_costs[experiment_world.goal])
print()
print(experiment_world.draw(path=experiment_path))

A* cost: 9
A* steps: 34
Zero-heuristic steps: 65
Same shortest cost: True

....#....
.##***##.
.S**#***G
.##...##.
.........


## Visual Trace + Rigor Studio

**Problem frame.** Use a heuristic to guide shortest-path search toward a goal.

**Interactive animation target.** Animate open set, closed set, g-score, h-score, and f-score layers.

**Correctness handle.** With an admissible and consistent heuristic, the first goal popped has optimal cost.

**Complexity handle.** Worst-case exponential in path depth, but often far less search than uninformed methods.

**Failure mode to test.** An overconfident heuristic can skip the true shortest path.

**Studio task.** Weaken and strengthen the heuristic, then compare the number of expanded nodes.


In [ ]:
from pathlib import Path
import sys

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "courseware").exists():
        sys.path.insert(0, str(candidate))
        break

from courseware import AlgorithmPlayer, AlgorithmTrace, TraceStep, render_trace_table

# Convert the implementation above into snapshots:
# trace = AlgorithmTrace("Topic trace")
# trace.append("start", {"your_state": ...}, "What changed?", invariant="What remains true?")
# AlgorithmPlayer(trace, your_renderer).display()
print("Use AlgorithmTrace to turn this notebook's algorithm into a step-by-step visual player.")
